In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
NCF_results=pd.read_csv("/home/mvarasteh/ProvidersExplantion/backward_selection_results_MLP_LASTFM.csv", index_col=0)
BPR_results=pd.read_csv("/home/mvarasteh/ProvidersExplantion/backward_selection_results_BPR_LASTFM.csv", index_col=0)
VAE_results=pd.read_csv("/home/mvarasteh/ProvidersExplantion/backward_selection_results_VAE_LASTFM.csv", index_col=0)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

y_NCF = NCF_results['R2'].to_list()
y_BPR = BPR_results['R2'].to_list()
y_VAE = VAE_results['R2'].to_list()

x_labels_NCF = NCF_results['R2'].index.to_list()
x_labels_BPR = BPR_results['R2'].index.to_list()
x_labels_VAE = VAE_results['R2'].index.to_list()

offset = 0.22 # horizontal nudge to avoid overlap

models = [
    ('NCF', y_NCF, x_labels_NCF, -offset),
    ('BPR', y_BPR, x_labels_BPR,  0),
    ('VAE', y_VAE, x_labels_VAE, +offset),
]

fig, ax = plt.subplots(figsize=(10, 6))

for model_name, y_vals, x_labels, shift in models:
    x_nums = np.array(range(len(x_labels))) + shift
    ax.plot(x_nums, y_vals, marker='o', label=model_name, linewidth=2, markersize=20)
    for i, (xi, yi) in enumerate(zip(x_labels, y_vals)):
        ax.annotate(xi,
                    xy=(i + shift, yi),
                    ha='center', va='center',
                    fontsize=6, fontweight='bold', color='white',)

max_len = max(len(x_labels_NCF), len(x_labels_BPR), len(x_labels_VAE))
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title('Backward Selection - Last.fm', fontsize=14, fontweight='bold')
ax.set_xlabel('Rank of Features Removed', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_xticks(range(max_len))
ax.set_xticklabels(range(max_len))
ax.legend(fontsize=11)
plt.savefig("/home/mvarasteh/ProvidersExplantion/backward_selection_plot_LASTFM.png", dpi=300)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Align all models to a common feature order
all_features_NCF = NCF_results['R2'].index.to_list()
all_features_BPR = BPR_results['R2'].index.to_list()
all_features_VAE = VAE_results['R2'].index.to_list()

y_NCF = NCF_results['R2'].to_list()
y_BPR = BPR_results['R2'].to_list()
y_VAE = VAE_results['R2'].to_list()

models = ['NCF', 'BPR', 'VAE']
all_labels = [all_features_NCF, all_features_BPR, all_features_VAE]
all_values = [y_NCF, y_BPR, y_VAE]

# Find the maximum length for padding
max_len = max(len(l) for l in all_labels)

# Pad shorter sequences with NaN
data = []
col_labels = []
for labels, values in zip(all_labels, all_values):
    padded = values + [np.nan] * (max_len - len(values))
    data.append(padded)
    col_labels = labels if len(labels) == max_len else col_labels

data = np.array(data)

fig, ax = plt.subplots(figsize=(12, 3))

cmap = plt.cm.RdYlGn  # red = low, green = high
cmap.set_bad(color='lightgray')  # NaN cells shown as gray

im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Add R² value annotations inside each cell
for i in range(len(models)):
    for j in range(max_len):
        val = data[i, j]
        if not np.isnan(val):
            text_color = 'black' if 0.3 < val < 0.85 else 'white'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color=text_color)

# X-axis: feature name + step number
x_tick_labels = [f'{j}\n({col_labels[j]})' for j in range(max_len)]

ax.set_xticks(range(max_len))
ax.set_xticklabels(x_tick_labels, fontsize=9)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=11, fontweight='bold')

ax.set_title('Feature Ablation Study - Last.fm', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Features Removed (Feature Name)', fontsize=11)

cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label('R²', fontsize=11)

plt.tight_layout()
plt.savefig('ablation_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

y_NCF = NCF_results['R2'].to_list()
y_BPR = BPR_results['R2'].to_list()
y_VAE = VAE_results['R2'].to_list()

x_labels_NCF = NCF_results['R2'].index.to_list()
x_labels_BPR = BPR_results['R2'].index.to_list()
x_labels_VAE = VAE_results['R2'].index.to_list()

# Use the longest label list for x-axis
all_labels = [x_labels_NCF, x_labels_BPR, x_labels_VAE]
col_labels = max(all_labels, key=len)
max_len = len(col_labels)

# Pad shorter sequences with NaN
def pad(values, length):
    return values + [np.nan] * (length - len(values))

y_NCF_p = pad(y_NCF, max_len)
y_BPR_p = pad(y_BPR, max_len)
y_VAE_p = pad(y_VAE, max_len)

x = np.arange(max_len)
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

bars_NCF = ax.bar(x - width, y_NCF_p, width, label='NCF', color='steelblue', edgecolor='white')
bars_BPR = ax.bar(x,         y_BPR_p, width, label='BPR', color='darkorange', edgecolor='white')
bars_VAE = ax.bar(x + width, y_VAE_p, width, label='VAE', color='seagreen', edgecolor='white')

# Add R² value on top of each bar
for bars, values in [(bars_NCF, y_NCF_p), (bars_BPR, y_BPR_p), (bars_VAE, y_VAE_p)]:
    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f'{val:.2f}',
                    ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title('Feature Ablation Study - Last.fm', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Features Removed', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f'{i}\n({col_labels[i]})' for i in range(max_len)], fontsize=9)
ax.legend(fontsize=11)
ax.set_ylim(bottom=min(min(y_NCF), min(y_BPR), min(y_VAE)) - 0.1)
plt.tight_layout()
plt.savefig('ablation_grouped_bar.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
x=NCF_results['R2'].index.to_list()
y_NCF=NCF_results['R2'].to_list()
x_labels = x
x_nums = list(range(len(x_labels)))  # [0, 1, 2, 3, 4, 5, 6, 7]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(x_nums, y_NCF, marker='o', label='NCF', linewidth=2, markersize=20)

# Add feature name inside each circle
for i, (xi, yi) in enumerate(zip(x_labels, y_NCF)):
    ax.annotate(xi,
                xy=(i, yi),
                ha='center', va='center',
                fontsize=6, fontweight='bold', color='white')

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.set_title('Feature Ablation Study', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Features Removed', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_xticks(x_nums)
ax.set_xticklabels(x_nums)  # shows 0, 1, 2, 3, 4, 5, 6, 7
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
NCF_results=NCF_results.drop(columns=["MSE"])
BPR_results=BPR_results.drop(columns=["MSE", "feature_added"])
VAE_results=VAE_results.drop(columns=["MSE", "feature_added"])


In [ ]:
BPR_results=BPR_results.rename(columns={"R2":"BPR"})
VAE_results=VAE_results.rename(columns={"R2":"VAE"})
NCF_results=NCF_results.rename(columns={"R2":"NCF"})

In [ ]:
All_results=pd.concat([NCF_results,BPR_results,VAE_results], axis=1)

In [ ]:
All_results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np



# 2. Create Cumulative Labels for the X-axis
labels = [r'$\beta_1$', r'$\beta_1+\beta_2$'] + [rf'$+\beta_{i}$' for i in range(3, 9)]
x_indices = np.arange(len(labels))

# 3. Setup Plotting Style
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7), dpi=300)

# 4. Plot each model with specific markers
ax.plot(x_indices, All_results['BPR'], marker='o', markersize=8, linewidth=3, label='BPR', color='#1f77b4')
ax.plot(x_indices, All_results['VAE'], marker='s', markersize=8, linewidth=3, label='VAE', color='#ff7f0e')
ax.plot(x_indices, All_results['NCF'], marker='^', markersize=9, linewidth=3, label='NCF', color='#2ca02c')

# 5. Add "Saturation Point" Annotation (placed around B4 where curves flatten)

# 6. Formatting
#ax.set_title(r'Cumulative Feature Impact on Fidelity ($R^2$)', 
            # fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Cumulative Feature Set', fontsize=16, labelpad=10)
ax.set_ylabel(r'Surrogate Model $R^2$ Score', fontsize=16, labelpad=10)
ax.tick_params(axis='y', labelsize=16)
ax.set_xticks(x_indices)
ax.set_xticklabels(labels, fontsize=16)
ax.set_ylim(-0.2, 1.0) # Adjusted to show the detail while cutting off the negative NCF tail

# Legend styling
ax.legend(loc='lower right', frameon=True, shadow=True, fontsize=16, borderpad=1)
plt.savefig("Cumulative feature impact")
plt.tight_layout()
plt.show()


In [ ]:
BPR_exp=pd.read_csv("/home/mvarasteh/ProvidersExplantion/exp_df_BPR_ML1M.csv", index_col=0)
NCF_exp=pd.read_csv("/home/mvarasteh/ProvidersExplantion/exp_df_MLP_ML1M.csv", index_col=0)
VAE_exp=pd.read_csv("/home/mvarasteh/ProvidersExplantion/exp_df_VAE_ML1M.csv", index_col=0)

In [ ]:
sorted_BPR_exp=np.sort(BPR_exp["exp"].values)[::-1]
sorted_NCF_exp=np.sort(NCF_exp["exp"].values)[::-1]

sorted_VAE_exp=np.sort(VAE_exp["exp"].values)[::-1]

num_itms=3952

In [ ]:
sorted_BPR_exp=np.pad(sorted_BPR_exp, (0, num_itms-len(sorted_BPR_exp)), mode='constant', constant_values=0 )
sorted_NCF_exp=np.pad(sorted_NCF_exp, (0, num_itms-len(sorted_NCF_exp)), mode='constant', constant_values=0 )
sorted_VAE_exp=np.pad(sorted_VAE_exp, (0, num_itms-len(sorted_VAE_exp)), mode='constant', constant_values=0 )

In [ ]:
plt.plot(sorted_BPR_exp, label="BPR", linewidth=2, color='#1f77b4')
plt.plot(sorted_NCF_exp,label='NCF', linewidth=2, color='#ff7f0e' )
plt.plot(sorted_VAE_exp, label='VAE', linewidth=2, color='#2ca02c')
plt.xlabel('Items ', fontsize=14)
plt.ylabel('Exposure', fontsize=14)
plt.legend(fontsize=14, frameon=True, shadow=True)
plt.title("Movielens")
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.savefig("exposure.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))

# Use a small offset to prevent log(0) issues for the long tail
offset = 1e-1
x_axis = np.arange(3707)
# Plotting with refined styles
plt.plot(x_axis, sorted_BPR_exp + offset, 
         label='BPR', linewidth=2.5, color='#1f77b4')
plt.plot(x_axis, sorted_NCF_exp + offset, 
         label='NCF', linewidth=2.5, color='#ff7f0e')
plt.plot(np.arange(len(sorted_VAE_exp)), sorted_VAE_exp + offset, 
         label='VAE', linewidth=2.5, color='#2ca02c')

#plt.xscale('log') 
#plt.yscale('log') 

# Labels and Title
plt.xlabel('Items', fontsize=12, fontweight='bold')
plt.ylabel('Exposure Frequency', fontsize=12, fontweight='bold')
plt.title('Item Exposure Distribution', fontsize=14, pad=15)

# Adding the grid back for better readability of the log steps
plt.grid(True, which="both", ls="-", alpha=0.2)
#plt.xlim([-1000,1000])
plt.legend(frameon=True, shadow=True)
plt.tight_layout()

# Save for your two-column paper (single column width)
plt.savefig('exposure.pdf', bbox_inches='tight', dpi=300)
plt.show()